In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import polars as pl

from config import (
    SAML_D_VALIDATION_START,
    SAML_D_TEST_START,
    SAML_D_TEST_END_EXCLUSIVE
)

from src.data.loader import SAMLDDataLoader

from src.graph.temporal_split import SAMLDTemporalSplitter
from src.graph.builder import SAMLDGraphTableBuilder

In [2]:
### load standardized transactions
# create loader instance
loader = SAMLDDataLoader()

# lazily scan transactions from interim Parquet dataset
transactions = loader.scan_interim()

# create temporal splitter instance
temporal_splitter = SAMLDTemporalSplitter(
    transactions = transactions,
    validation_start = SAML_D_VALIDATION_START,
    test_start = SAML_D_TEST_START,
    test_end_exclusive = SAML_D_TEST_END_EXCLUSIVE
)

# create graph builder instance
graph_builder = SAMLDGraphTableBuilder(
    transactions = transactions
)

In [3]:
### validation snapshots
# build account-node table
validation_nodes = graph_builder.build_node_table(
    cutoff = SAML_D_VALIDATION_START
)

# build aggregated edge per directed account pair
validation_edges = graph_builder.build_edge_table(
    cutoff = SAML_D_VALIDATION_START
)

# inspect schemas
display(validation_nodes.collect_schema())
display(validation_edges.collect_schema())

Schema([('node_id', UInt64),
        ('account', String),
        ('first_seen_timestamp', Datetime(time_unit='us', time_zone=None)),
        ('last_seen_timestamp', Datetime(time_unit='us', time_zone=None)),
        ('outgoing_transaction_count', UInt64),
        ('incoming_transaction_count', UInt64),
        ('account_transaction_event_count', UInt64),
        ('out_degree', UInt64),
        ('in_degree', UInt64),
        ('total_directed_degree', UInt64),
        ('bank_location_count', UInt64),
        ('is_multi_location', Boolean),
        ('has_outgoing_activity', Boolean),
        ('has_incoming_activity', Boolean)])

Schema([('edge_id', UInt64),
        ('source_node_id', UInt64),
        ('target_node_id', UInt64),
        ('sender_account', String),
        ('receiver_account', String),
        ('transaction_count', UInt64),
        ('count_weight', Float64),
        ('log_count_weight', Float64),
        ('first_transaction_timestamp',
         Datetime(time_unit='us', time_zone=None)),
        ('last_transaction_timestamp',
         Datetime(time_unit='us', time_zone=None)),
        ('active_day_count', UInt64),
        ('payment_type_count', UInt64),
        ('payment_currency_count', UInt64),
        ('received_currency_count', UInt64),
        ('cross_currency_transaction_count', UInt64),
        ('cross_currency_share', Float64),
        ('is_repeated_edge', Boolean),
        ('is_self_loop', Boolean)])

In [4]:
### inspect validation samples
# node samples
validation_node_samples = (
    validation_nodes
    .head(10)
    .collect(
        engine = 'streaming'
    )
)

# edge samples
validation_edge_samples = (
    validation_edges
    .head(10)
    .collect(
        engine = 'streaming'
    )
)

# inspect samples
display(validation_node_samples)
display(validation_edge_samples)

node_id,account,first_seen_timestamp,last_seen_timestamp,outgoing_transaction_count,incoming_transaction_count,account_transaction_event_count,out_degree,in_degree,total_directed_degree,bank_location_count,is_multi_location,has_outgoing_activity,has_incoming_activity
u64,str,datetime[μs],datetime[μs],u64,u64,u64,u64,u64,u64,u64,bool,bool,bool
0,"""1000005344""",2023-02-09 00:56:06,2023-02-09 19:32:31,0,13,13,0,1,1,1,false,false,true
1,"""1000038979""",2022-11-29 17:06:00,2023-05-09 10:16:15,0,10,10,0,1,1,1,false,false,true
2,"""100004626""",2023-03-08 08:19:26,2023-03-08 23:39:14,12,0,12,1,0,1,1,false,true,false
3,"""1000065088""",2022-10-22 13:12:47,2023-05-23 13:34:12,0,4,4,0,1,1,1,false,false,true
4,"""1000065782""",2022-11-10 06:38:22,2023-03-13 00:10:20,0,5,5,0,1,1,1,false,false,true
5,"""1000079875""",2022-11-11 17:29:23,2023-05-13 05:23:13,0,6,6,0,1,1,1,false,false,true
6,"""1000105878""",2023-04-20 06:09:58,2023-04-20 23:22:48,12,0,12,1,0,1,1,false,true,false
7,"""1000118981""",2023-05-22 11:24:18,2023-05-22 11:24:18,0,1,1,0,1,1,1,false,false,true
8,"""1000120075""",2023-02-20 11:13:01,2023-02-20 22:23:31,12,0,12,1,0,1,1,false,true,false


edge_id,source_node_id,target_node_id,sender_account,receiver_account,transaction_count,count_weight,log_count_weight,first_transaction_timestamp,last_transaction_timestamp,active_day_count,payment_type_count,payment_currency_count,received_currency_count,cross_currency_transaction_count,cross_currency_share,is_repeated_edge,is_self_loop
u64,u64,u64,str,str,u64,f64,f64,datetime[μs],datetime[μs],u64,u64,u64,u64,u64,f64,bool,bool
0,2,425199,"""100004626""","""618248540""",12,12.0,2.564949,2023-03-08 08:19:26,2023-03-08 23:39:14,1,3,1,1,0,0.0,true,false
1,6,351567,"""1000105878""","""5286213996""",12,12.0,2.564949,2023-04-20 06:09:58,2023-04-20 23:22:48,1,4,1,1,0,0.0,true,false
2,8,241773,"""1000120075""","""3945413235""",12,12.0,2.564949,2023-02-20 11:13:01,2023-02-20 22:23:31,1,4,2,1,1,0.083333,true,false
3,9,732043,"""1000121944""","""9938882588""",3,3.0,1.386294,2023-02-08 10:54:19,2023-03-20 07:34:18,3,1,1,1,0,0.0,true,false
4,13,69909,"""1000200981""","""1855234502""",4,4.0,1.609438,2022-11-01 18:46:52,2023-02-20 20:10:41,4,1,1,1,0,0.0,true,false
5,13,88448,"""1000200981""","""2078610719""",2,2.0,1.098612,2023-05-14 17:35:49,2023-05-19 09:20:35,2,1,1,1,2,1.0,true,false
6,13,109501,"""1000200981""","""2334927389""",10,10.0,2.397895,2022-10-25 16:53:17,2023-05-01 03:41:22,10,4,1,1,0,0.0,true,false
7,13,285786,"""1000200981""","""4483257877""",4,4.0,1.609438,2022-10-29 10:39:00,2023-04-10 16:43:00,4,2,1,1,0,0.0,true,false
8,13,317985,"""1000200981""","""4876055248""",14,14.0,2.70805,2022-10-15 13:44:55,2023-05-24 04:01:04,13,4,1,1,0,0.0,true,false


In [5]:
### assert label leakage
forbidden_graph_columns = {
    'is_laundering',
    'laundering_type',
    'known_before_window',
    'is_new_suspicious_account',
    'is_rankable_new_suspicious_account'
}

node_columns = set(
    validation_nodes
    .collect_schema()
    .names()
)

edge_columns = set(
    validation_edges
    .collect_schema()
    .names()
)

assert(
    forbidden_graph_columns.isdisjoint(node_columns)
)

assert(
    forbidden_graph_columns.isdisjoint(edge_columns)
)

In [6]:
### graph summary
# validation snapshot graph summary
validation_graph_summary = graph_builder.get_snapshot_summary(
    cutoff = SAML_D_VALIDATION_START
)

with pl.Config(
    tbl_cols = -1,
    tbl_width_chars = 240
):
    display(validation_graph_summary)

snapshot_cutoff,historical_transaction_count,edge_transaction_count,transaction_reconciliation_difference,node_count,unique_account_count,unique_node_id_count,minimum_node_id,maximum_node_id,edge_count,unique_directed_pair_count,repeated_edge_count,repeated_edge_share,maximum_transactions_per_edge,mean_transactions_per_edge,self_loop_transaction_count,self_loop_edge_count,missing_source_node_id_count,missing_target_node_id_count,sender_and_receiver_node_count,sender_only_node_count,receiver_only_node_count,multi_location_node_count
datetime[μs],u64,u64,i64,u64,u64,u64,u64,u64,u64,u64,u64,f64,u64,f64,u64,u64,u64,u64,u64,u64,u64,u64
2023-06-01 00:00:00,7048788,7048788,0,736925,736925,736925,0,736924,761365,761365,666717,0.875686,62,9.258093,0,0,0,0,78177,158717,500031,5336


In [7]:
### integrity assertions
validation_summary_row = (
    validation_graph_summary
    .row(
        index = 0,
        named = True
    )
)

assert (
    validation_summary_row['historical_transaction_count'] == validation_summary_row['edge_transaction_count']
)

assert (
    validation_summary_row['transaction_reconciliation_difference'] == 0
)

assert (
    validation_summary_row['node_count'] == validation_summary_row['unique_account_count']
)

assert (
    validation_summary_row['node_count'] == validation_summary_row['unique_node_id_count']
)

assert (
    validation_summary_row['minimum_node_id'] == 0
)

assert (
    validation_summary_row['maximum_node_id'] == validation_summary_row['node_count'] - 1
)

assert(
    validation_summary_row['edge_count'] == validation_summary_row['unique_directed_pair_count']
)

assert (
    validation_summary_row['missing_source_node_id_count'] == 0
)

assert (
    validation_summary_row['missing_target_node_id_count'] == 0
)

In [8]:
### reconcile graph nodes with temporal populations
# historical seed count
validation_seed_count = (
    temporal_splitter
    .build_known_suspicious_accounts(
        cutoff = SAML_D_VALIDATION_START
    )
    .select(
        pl.len()
        .alias(
            'seed_count'
        )
    )
    .collect(engine = 'streaming')
    .item()
)

# historical non-seed candidates
validation_candidate_count = (
    temporal_splitter
    .build_candidate_accounts(
        cutoff = SAML_D_VALIDATION_START
    )
    .select(
        pl.len()
        .alias(
            'candidate_count'
        )
    )
    .collect(engine = 'streaming')
    .item()
)

# historical node count
validation_node_count = validation_summary_row['node_count']

print(
    f'Validation seed count: {validation_seed_count:,}\n'
    f'Validation non-seed count: {validation_candidate_count:,}\n'
    f'Validation node count: {validation_node_count:,}'
)

assert(
    validation_node_count == (validation_seed_count + validation_candidate_count)
)

Validation seed count: 5,737
Validation non-seed count: 731,188
Validation node count: 736,925


In [9]:
### check if all rankable targets map to nodes
# rankable validation accounts
rankable_validation_accounts = (
    temporal_splitter
    .build_future_suspicious_accounts(
        window_start = SAML_D_VALIDATION_START,
        window_end_exclusive = SAML_D_TEST_START
    )
    .filter(
        pl.col('is_rankable_new_suspicious_account')
    )
    .select('account')
)

# rankable target mappings
rankable_target_mapping_summary = (
    rankable_validation_accounts
    .join(
        validation_nodes
        .select(
            [
                'account',
                'node_id'
            ]
        ),
        on = 'account',
        how = 'left'
    )
    .select(
        [
            # rankable target count
            pl.len()
            .cast(pl.UInt64)
            .alias(
                'rankable_target_count'
            ),
            # missing target node id count
            pl.col('node_id')
            .null_count()
            .cast(pl.UInt64)
            .alias(
                'missing_target_node_id_count'
            )
        ]
    )
    .collect(engine = 'streaming')
)

display(rankable_target_mapping_summary)

assert (
    rankable_target_mapping_summary['missing_target_node_id_count'][0] == 0
)

rankable_target_count,missing_target_node_id_count
u64,u64
678,0


**Graph Builder Summary**

$G_{validation} = (V_{history}, E_{history})$

The edge table support three PageRank variants:
- Unweighted PageRank
- Count-weighted PageRank
- Log-count weighted PageRank

Interpretation of the graph structure:
- ~7m transactions are reduced to 761k directed edges
- Repeated edges: 666k
- Single-transaction edges: 94k
- Repeated edge share: 87.57%
- Maximum transactions per edge: 62

Above metrics support the decision of aggregating transaction records before implementing PageRank

Account-role counts:
- Sender and receiver: 78k -> 10.61% share
- Sender only: 158k -> 21.54% share
- Receiver only: 500k -> 67.85% share
- Total: 736k

~500k accounts are receiver-only, these are dangling nodes in the sender-to-receiver graph, representing 67.85% of all nodes

Density perspective:
- Density = total edges / (total nodes x (total nodes - 1)) = $761k / 736k^{2}$ -> extreme sparse graph

